## Counting Model

In [2]:
import torch
import itertools

In [3]:
words = open("Names.txt", "r").read().splitlines()

In [4]:
letters = sorted(set("".join(words)))

ltr_to_idx = {ltr: idx+1 for idx, ltr in enumerate(letters)}
ltr_to_idx["."] = 0

idx_to_ltr = {idx: ltr for ltr, idx in ltr_to_idx.items()}

In [5]:
# bigrams = {}
#
# for word in words:
#     word = ["."] + list(word) + ["."]
#
#     for ltr1, ltr2 in zip(word, word[1:]):
#         bigram = (ltr1, ltr2)
#
#         bigrams[bigram] = bigrams.get(bigram, 0) + 1

In [6]:
Bigrams = torch.zeros((27, 27), dtype=torch.int)

for word in words:
    word = ["."] + list(word) + ["."]

    for ltr1, ltr2 in zip(word, word[1:]):
        row_idx = ltr_to_idx[ltr1]
        col_idx = ltr_to_idx[ltr2]

        Bigrams[row_idx, col_idx] += 1

In [7]:
Bigram_distribution = (Bigrams+1)/ (Bigrams+1).sum(dim=1, keepdim=True)

In [8]:
sampler = torch.Generator().manual_seed(0)

for turn in range(len(words)):

    ltrs = []
    index = 0

    while True:

        current_distribution = Bigram_distribution[index]
        index = torch.multinomial(current_distribution, num_samples=1, replacement=True, generator=sampler).item()
        ltrs.append(idx_to_ltr[index])

        if index == 0:
            break

    print("".join(ltrs))

kedan.
frenakey.
aderyna.
salyler.
rion.
kefwajusi.
ziala.
kanocikhilinileea.
m.
cyl.
kar.
re.
jesho.
jq.
shriaz.
eanzen.
k.
dyaha.
jam.
sohaynte.
yeychyariod.
aza.
ama.
kus.
kaynalah.
drendadiey.
bour.
kne.
jan.
trmialyaha.
mande.
konle.
cezioraryl.
mara.
ka.
kiah.
re.
ale.
brtorkaherey.
deneckene.
h.
stiobonylana.
selobrllee.
k.
jarrri.
bayzanandize.
je.
kienvleelicinghll.
mson.
diarale.
m.
dieiomanntadeleshlays.
sadmista.
ke.
te.
l.
njertha.
h.
caonee.
fri.
bre.
teyeillubayse.
len.
marala.
vynadaser.
ji.
abddonaniarexyd.
jadoprteve.
mnath.
negaryaynahlecrvion.
e.
tayaiebijalarielilynadimamacyaha.
nayamaelavin.
t.
s.
d.
raltelamrin.
kore.
tehtopk.
va.
bo.
s.
okashy.
enn.
asasigtyringrlewismmelyll.
khaysta.
auelalen.
ah.
an.
harei.
gelomielas.
cariadencazilalero.
iarteallelir.
booucerilet.
tebrady.
aharmb.
j.
ahi.
toreer.
jovil.
java.
junza.
mi.
daamareemarck.
kriee.
liamoenon.
korearigresasulagrf.
ahy.
ka.
juvavacquso.
zillden.
tajan.
vi.
tonn.
laleun.
hari.
narladahauch.
mam.
rnn.
z

In [9]:
loss = 0
count = 0

for word in words:
    word = ["."] + list(word) + ["."]

    for ltr1, ltr2 in zip(word, word[1:]):
        indices = (ltr_to_idx[ltr1], ltr_to_idx[ltr2])
        bigram_probability = Bigram_distribution[ltr_to_idx[ltr1], ltr_to_idx[ltr2]]

        log_probability = torch.log(bigram_probability)
        loss += -log_probability

        count += 1

        # print(f"{ltr1}{ltr2} has probability={bigram_probability:.4f} and loss={log_probability:.4f}")


print(f"Total loss is: {(loss/count):.4f}")

Total loss is: 2.4544


## Neural Net Model

In [10]:
input_indices = []
output_indices = []

for word in words:
    word = ["."] + list(word) + ["."]

    for ltr1, ltr2 in zip(word, word[1:]):
        input_idx = ltr_to_idx[ltr1]
        output_idx = ltr_to_idx[ltr2]

        input_indices.append(input_idx)
        output_indices.append(output_idx)

inputs = torch.tensor(input_indices, dtype=torch.int)
outputs = torch.tensor(output_indices, dtype=torch.int)

extra_in = []
extra_out = []

for i, j in itertools.product(range(27), range(27)):
    extra_in.append(i)
    extra_out.append(j)

train_inputs = torch.cat([inputs, torch.tensor(extra_in)])
train_outputs = torch.cat([outputs, torch.tensor(extra_out)])

In [11]:
sampler_nn = torch.Generator().manual_seed(0)
Weight = torch.randn((27, 27), generator=sampler_nn, requires_grad=True)

In [12]:
for turn in range(1000):
    logits = Weight[train_inputs]
    probabilities = logits.exp() / logits.exp().sum(dim=1, keepdim=True)

    nn_loss = -probabilities[torch.arange(len(train_inputs)), train_outputs].log().mean()
    print(f"Loss at {turn}'th turn is {nn_loss.item():.4f}")

    Weight.grad = None
    nn_loss.backward()

    with torch.no_grad():
        Weight += -50 * Weight.grad

Loss at 0'th turn is 3.7678
Loss at 1'th turn is 3.3596
Loss at 2'th turn is 3.1521
Loss at 3'th turn is 3.0227
Loss at 4'th turn is 2.9337
Loss at 5'th turn is 2.8662
Loss at 6'th turn is 2.8132
Loss at 7'th turn is 2.7712
Loss at 8'th turn is 2.7374
Loss at 9'th turn is 2.7099
Loss at 10'th turn is 2.6871
Loss at 11'th turn is 2.6680
Loss at 12'th turn is 2.6517
Loss at 13'th turn is 2.6376
Loss at 14'th turn is 2.6252
Loss at 15'th turn is 2.6143
Loss at 16'th turn is 2.6046
Loss at 17'th turn is 2.5959
Loss at 18'th turn is 2.5881
Loss at 19'th turn is 2.5811
Loss at 20'th turn is 2.5747
Loss at 21'th turn is 2.5690
Loss at 22'th turn is 2.5637
Loss at 23'th turn is 2.5589
Loss at 24'th turn is 2.5545
Loss at 25'th turn is 2.5504
Loss at 26'th turn is 2.5466
Loss at 27'th turn is 2.5431
Loss at 28'th turn is 2.5398
Loss at 29'th turn is 2.5368
Loss at 30'th turn is 2.5339
Loss at 31'th turn is 2.5312
Loss at 32'th turn is 2.5287
Loss at 33'th turn is 2.5264
Loss at 34'th turn is 2.

In [13]:
logits = Weight[inputs]
probabilities = logits.exp() / logits.exp().sum(dim=1, keepdim=True)
eval_loss = -probabilities[torch.arange(len(inputs)), outputs].log().mean()
print(f"NN loss on original data: {eval_loss.item():.4f}")

NN loss on original data: 2.4557
